
# FinTech & BFSI: UPI Fraud Ring & Merchant Analytics

**Business problem.** A National Payments Authority needs to analyze micro‑transaction
graphs to identify **circular money‑laundering rings**, **synthetic identity fraud**,
and **compromised merchant accounts**, from high‑velocity UPI logs that are full of
missing UTR numbers, mismatched PAN/Aadhaar formats, OCR‑style errors, and currency
symbols embedded in numeric columns.

**Datasets used**
| File | Grain | Rows (raw) |
|---|---|---|
| `track1_upi_transactions.csv` | 1 row = 1 UPI transaction | 20,400 |
| `track1_kyc_records.csv` | 1 row = 1 customer KYC record | 36,400 |
| `track1_merchants_master.csv` | 1 row = 1 merchant | 6,210 |
| `track1_chargebacks.json` | 1 row = 1 dispute / chargeback complaint | 2,884 |


In [ ]:

import os
import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_DIR = '.' if os.path.exists('track1_upi_transactions.csv') else ('DATA' if os.path.exists('DATA/track1_upi_transactions.csv') else '../DATA')

TXN_PATH   = os.path.join(DATA_DIR, 'track1_upi_transactions.csv')
KYC_PATH   = os.path.join(DATA_DIR, 'track1_kyc_records.csv')
MERCH_PATH = os.path.join(DATA_DIR, 'track1_merchants_master.csv')
CBK_PATH   = os.path.join(DATA_DIR, 'track1_chargebacks.json')


## 1. Load & profile the raw data

In [ ]:

txns_raw = pd.read_csv(TXN_PATH, dtype=str)
kyc_raw = pd.read_csv(KYC_PATH, dtype=str)
merch_raw = pd.read_csv(MERCH_PATH, dtype=str)
with open(CBK_PATH) as f:
    cbk_raw = pd.DataFrame(json.load(f))

print('transactions :', txns_raw.shape)
print('kyc records  :', kyc_raw.shape)
print('merchants    :', merch_raw.shape)
print('chargebacks  :', cbk_raw.shape)


In [ ]:

txns_raw.head()


In [ ]:

# Quick null / duplicate profile of every raw table BEFORE any cleaning
for name, df in [('transactions', txns_raw), ('kyc', kyc_raw), ('merchants', merch_raw), ('chargebacks', cbk_raw)]:
    print(f'--- {name} ---')
    print('nulls per column:')
    print(df.isna().sum()[df.isna().sum() > 0])
    print('exact full-row duplicates:', df.astype(str).duplicated().sum())
    print()



A first look already shows the mess we've been promised: currency symbols and commas
mixed into `amount` / `monthly_income` / `disputed_amount`, at least six different
timestamp formats, a dozen spellings each for `status`, `kyc_status`, `merchant_status`,
`severity`, `resolution_status`, `reason_code`, differently‑formatted `user_id` /
`merchant_id` / `txn_id` values, OCR‑style PAN/Aadhaar noise, and inconsistent city names.


In [ ]:

print(txns_raw['status'].value_counts())
print()
print(txns_raw[['amount','utr','mcc']].sample(8, random_state=1))
print()
print(txns_raw['timestamp'].sample(8, random_state=1).tolist())


## 2. Cleaning utilities

Each helper below encodes one cleaning decision, verified against the actual data (see exploration in the accompanying analysis) rather than assumed.

In [ ]:

def normalize_id(value, prefix, width):
    '''Standardize IDs like USR12345 / usr-12345 / USR 12345 / 12345 -> PREFIX + zero-padded digits.
    `width` is the expected digit-count for this id family, confirmed from the data:
    USR -> 5 digits, MCH -> 4 digits, TXN -> 8 digits.'''
    if pd.isna(value):
        return np.nan
    s = str(value).strip().upper()
    s = re.sub(rf'^{prefix}[\s_\-]*', '', s)
    digits = re.sub(r'\D', '', s)
    if digits == '':
        return np.nan
    return f'{prefix}{digits.zfill(width)}'


def clean_amount(value):
    '''Strip currency symbols (Rs., INR, ₹), thousands commas, and 'k' shorthand
    (e.g. '27.3k' -> 27300) from a monetary field and return a float.
    Negative raw values are treated as sign/data-entry errors (there is no legitimate
    concept of a negative UPI transaction amount, income, or ticket size in this
    dataset) and are converted to their magnitude; `was_negative()` below lets us
    flag/audit exactly which rows this happened to instead of silently hiding it.'''
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if s == '' or s.lower() in ('na', 'n/a', 'null', 'none', 'not available'):
        return np.nan
    s = s.replace('₹', '').replace('Rs.', '').replace('Rs', '').replace('INR', '').strip()
    s = s.replace(',', '')
    k_suffix = s.lower().endswith('k')
    if k_suffix:
        s = s[:-1]
    s = re.sub(r'[^0-9.\-]', '', s)
    if s in ('', '-', '.', '-.'):
        return np.nan
    try:
        val = float(s)
    except ValueError:
        return np.nan
    if k_suffix:
        val *= 1000
    return abs(val)


def was_negative(value):
    '''Audit flag: did the raw string encode a negative amount before cleaning?'''
    if pd.isna(value):
        return False
    return str(value).strip().startswith('-')


In [ ]:

# --- Datetime parsing -------------------------------------------------------
# The four files mix (at least) six different timestamp conventions. Rather than
# letting pandas guess (which silently swaps day/month on ambiguous dates), we
# checked each observed pattern empirically -- by finding rows where one field of
# the pattern exceeds 12 -- and confirmed a *consistent* convention per pattern:
#
#   10-digit number                          -> Unix epoch seconds
#   YYYY-MM-DD[ HH:MM:SS]                     -> ISO
#   YYYY/MM/DD                                -> year-first slash
#   DD-Mon-YYYY[ HH:MM AM/PM]                 -> day-month name-year (unambiguous)
#   DD/MM/YYYY[ HH:MM(:SS)[AM/PM]]            -> day-first slash   (confirmed: first field reaches 31)
#   MM-DD-YYYY[ HH:MM:SS[AM/PM]]              -> month-first dash  (confirmed: first field maxes at 12)
#
# Anything that still doesn't parse, or parses to a date outside a sane
# 1920-2027 window, is treated as an impossible/corrupted timestamp -> NaT,
# instead of being coerced into a wrong-but-plausible-looking date.

UNIX_MIN, UNIX_MAX = 0, 2_000_000_000
DATE_SANE_MIN, DATE_SANE_MAX = pd.Timestamp('1920-01-01'), pd.Timestamp('2027-12-31')

_RE_ISO = re.compile(r'^\d{4}-\d{2}-\d{2}')
_RE_YMD_SLASH = re.compile(r'^\d{4}/\d{2}/\d{2}')
_RE_DMON = re.compile(r'^\d{1,2}-[A-Za-z]{3}-\d{4}')
_RE_SLASH_NUM = re.compile(r'^\d{1,2}/\d{1,2}/\d{4}')
_RE_DASH_NUM = re.compile(r'^\d{1,2}-\d{1,2}-\d{4}')
_RE_UNIX = re.compile(r'^-?\d{9,10}$')


def _sanity(dt):
    if pd.isna(dt):
        return pd.NaT
    if dt < DATE_SANE_MIN or dt > DATE_SANE_MAX:
        return pd.NaT
    return dt


def parse_datetime(value):
    if pd.isna(value):
        return pd.NaT
    s = str(value).strip()
    if s == '':
        return pd.NaT

    if _RE_UNIX.fullmatch(s):
        ts = int(s)
        if UNIX_MIN <= ts <= UNIX_MAX:
            return _sanity(pd.to_datetime(ts, unit='s', errors='coerce'))
        return pd.NaT

    if _RE_ISO.match(s):
        return _sanity(pd.to_datetime(s, errors='coerce'))
    if _RE_YMD_SLASH.match(s):
        return _sanity(pd.to_datetime(s, errors='coerce'))
    if _RE_DMON.match(s):
        return _sanity(pd.to_datetime(s, errors='coerce', dayfirst=True))
    if _RE_SLASH_NUM.match(s):
        return _sanity(pd.to_datetime(s, errors='coerce', dayfirst=True))
    if _RE_DASH_NUM.match(s):
        return _sanity(pd.to_datetime(s, errors='coerce', dayfirst=False))

    return _sanity(pd.to_datetime(s, errors='coerce'))


In [ ]:

# --- Categorical / status normalization maps --------------------------------
# Every map below was checked for 100% coverage against the actual distinct
# values in the raw columns (see exploration notes) before being finalized here.

def clean_text_upper_map(value, mapping, default=None):
    if pd.isna(value):
        return default
    key = str(value).strip().upper()
    return mapping.get(key, default if default is not None else key)

STATUS_MAP = {
    'S': 'SUCCESS', 'SUCCESS': 'SUCCESS', 'TXN_SUCCESS': 'SUCCESS', 'COMPLETED': 'SUCCESS',
    'F': 'FAILED', 'FAILED': 'FAILED', 'TXN_FAILED': 'FAILED', 'FAIL': 'FAILED', 'DECLINED': 'FAILED',
    'PENDING': 'PENDING', 'PROCESSING': 'PENDING', 'INITIATED': 'PENDING',
}
KYC_STATUS_MAP = {
    'V': 'VERIFIED', 'VERIFIED': 'VERIFIED', 'APPROVED': 'VERIFIED', 'DONE': 'VERIFIED', 'KYC_DONE': 'VERIFIED',
    'P': 'PENDING', 'PENDING': 'PENDING', 'IN_PROGRESS': 'PENDING', 'UNDER REVIEW': 'PENDING',
    'R': 'REJECTED', 'REJECTED': 'REJECTED', 'REJECT': 'REJECTED', 'FAILED': 'REJECTED',
}
RISK_SEGMENT_MAP = {'LOW': 'LOW', 'MEDIUM': 'MEDIUM', 'HIGH': 'HIGH', 'UNKNOWN': 'UNKNOWN'}
MERCHANT_STATUS_MAP = {
    'A': 'ACTIVE', 'ACTIVE': 'ACTIVE', 'ENABLED': 'ACTIVE', 'LIVE': 'ACTIVE',
    'I': 'INACTIVE', 'INACTIVE': 'INACTIVE', 'DISABLED': 'INACTIVE', 'CLOSED': 'INACTIVE',
    'S': 'SUSPENDED', 'SUSPENDED': 'SUSPENDED', 'HOLD': 'SUSPENDED', 'BLOCKED': 'SUSPENDED',
}
BUSINESS_TYPE_MAP = {
    'INDIVIDUAL': 'Individual', 'PARTNERSHIP': 'Partnership',
    'SOLE_PROPRIETOR': 'Sole Proprietor', 'SOLE-PROPRIETOR': 'Sole Proprietor', 'SOLE PROPRIETOR': 'Sole Proprietor',
    'PRIVATE_LIMITED': 'Private Limited', 'PRIVATE-LIMITED': 'Private Limited', 'PRIVATE LIMITED': 'Private Limited',
}
RESOLUTION_MAP = {
    'CLOSED': 'CLOSED', 'OPEN': 'OPEN', 'RESOLVED': 'RESOLVED', 'REJECTED': 'REJECTED',
    'PENDING BANK': 'PENDING_BANK', 'PENDING_BANK': 'PENDING_BANK',
    'IN PROGRESS': 'IN_PROGRESS', 'IN_PROGRESS': 'IN_PROGRESS', 'WIP': 'IN_PROGRESS',
}
SEVERITY_MAP = {
    'CRITICAL': 'CRITICAL', 'CRIT': 'CRITICAL', 'P1': 'CRITICAL',
    'HIGH': 'HIGH', 'H': 'HIGH', 'P2': 'HIGH',
    'MEDIUM': 'MEDIUM', 'M': 'MEDIUM', 'P3': 'MEDIUM',
    'LOW': 'LOW', 'L': 'LOW', 'P4': 'LOW',
}
REASON_CODE_MAP = {
    'CUSTOMER ISSUE': 'Customer Dispute', 'CUSTOMER DISPUTE': 'Customer Dispute', 'COMPLAINT': 'Customer Dispute',
    'DISPUTE RAISED': 'Customer Dispute',
    'DELIVERY ISSUE': 'Service Not Delivered', 'ITEM NOT RECEIVED': 'Service Not Delivered',
    'MERCHANT NOT DELIVERED': 'Service Not Delivered', 'NOT DELIVERED': 'Service Not Delivered',
    'NO SERVICE': 'Service Not Delivered', 'SERVICE FAILED': 'Service Not Delivered',
    'SERVICE NOT PROVIDED': 'Service Not Delivered', 'MERCHANT SERVICE ISSUE': 'Service Not Delivered',
    'NOT DONE BY ME': 'Service Not Delivered',
    'CHARGED TWICE': 'Duplicate Debit', 'DUPLICATE DEBIT': 'Duplicate Debit', 'DUP_DEBIT': 'Duplicate Debit',
    'DOUBLE DEBIT': 'Duplicate Debit',
    'EXTRA AMOUNT DEDUCTED': 'Wrong Amount', 'AMOUNT MISMATCH': 'Wrong Amount', 'WRONG AMOUNT': 'Wrong Amount',
    'INCORRECT AMOUNT': 'Wrong Amount',
    'ACCOUNT HACKED': 'Account Takeover', 'LOGIN COMPROMISED': 'Account Takeover', 'ATO': 'Account Takeover',
    'ACCOUNT TAKEOVER': 'Account Takeover',
    'UNAUTHORIZED TRANSACTION': 'Unauthorized Transaction', 'UNAUTH TXN': 'Unauthorized Transaction',
    'UNAUTHORIZED_TRANSACTION': 'Unauthorized Transaction', 'UNAUTHORISED': 'Unauthorized Transaction',
    'FRAUD': 'Suspected Fraud', 'FRAUD SUSPECTED': 'Suspected Fraud', 'SCAM': 'Suspected Fraud',
    'SUSPICIOUS TRANSACTION': 'Suspected Fraud',
}
CHANNEL_MAP = {
    'IVR': 'IVR', 'CHATBOT': 'Chatbot', 'EMAIL': 'Email', 'BRANCH': 'Branch',
    'APP': 'App', 'CALL CENTER': 'Call Center',
}

CITY_ALIASES = {
    'BOMBAY': 'Mumbai', 'MUMBAI': 'Mumbai', 'MUMBAY': 'Mumbai',
    'BLR': 'Bangalore', 'BANGALORE': 'Bangalore', 'BENGALURU': 'Bangalore',
    'JPR': 'Jaipur', 'JAIPUR': 'Jaipur',
    'DELHI': 'Delhi', 'DILLI': 'Delhi', 'NEW DELHI': 'Delhi',
    'ASR': 'Amritsar', 'AMRITSAR': 'Amritsar',
    'HYDERABAD': 'Hyderabad', 'HYD': 'Hyderabad',
    'MADRAS': 'Chennai', 'CHENNAI': 'Chennai',
    'CALCUTTA': 'Kolkata', 'KOLKATA': 'Kolkata',
    'LUCKNOW': 'Lucknow', 'LUDHIANA': 'Ludhiana', 'JALANDHAR': 'Jalandhar',
}

def clean_city(value):
    if pd.isna(value):
        return np.nan
    key = str(value).strip().upper()
    return CITY_ALIASES.get(key, str(value).strip().title())


In [ ]:

# --- UTR / MCC / PAN / Aadhaar ----------------------------------------------

def clean_utr(value):
    '''Keep only values matching the observed valid pattern UTR + 8-12 digits;
    anything else (blank, 'NA', malformed) becomes NaN and is flagged missing/invalid.'''
    if pd.isna(value):
        return np.nan
    s = str(value).strip().upper().replace(' ', '').replace('-', '')
    if s == '' or s in ('NA', 'NULL', 'NONE'):
        return np.nan
    return s if re.fullmatch(r'UTR\d{8,12}', s) else np.nan


def clean_mcc(value):
    '''Extract a clean 4-digit MCC from noisy strings like '05411', 'MCC-7011', '5311.0'.'''
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    digits = re.sub(r'\D', '', s.split('.')[0])
    if digits == '':
        return np.nan
    digits = digits.lstrip('0') or '0'
    return digits[-4:] if len(digits) >= 4 else np.nan


# The 10 numeric MCCs actually present in this dataset map 1:1 onto 10 business
# categories. Deriving `merchant_category` from the (numeric, low-noise) MCC code
# is far more reliable than trying to de-duplicate 30+ free-text spelling variants
# of the same category (e.g. 'HOTEL_LODGING' / 'Hotel' / 'Hotels' / 'Hospitality').
MCC_CATEGORY_MAP = {
    '4131': 'Transportation', '4814': 'Telecom', '5311': 'Department Store',
    '5411': 'Grocery', '5699': 'Apparel', '5812': 'Restaurant',
    '5912': 'Medical / Pharmacy', '5942': 'Book Store', '5999': 'Misc Retail',
    '7011': 'Hotel / Lodging',
}


def clean_pan(value):
    '''Standard PAN = 5 letters + 4 digits + 1 letter. Returns (standardized, is_valid);
    OCR-style truncation (missing trailing letter) etc. is preserved but flagged invalid
    rather than guessed at.'''
    if pd.isna(value):
        return np.nan, False
    s = re.sub(r'[\s\-]', '', str(value)).upper()
    if s == '':
        return np.nan, False
    return s, bool(re.fullmatch(r'[A-Z]{5}\d{4}[A-Z]', s))


def clean_aadhaar(value):
    '''Valid Aadhaar = 12 digits. Masked values ('XXXX-XXXX-1234') are unusable for
    matching/analytics and are treated as invalid/missing, not as real 12-digit IDs.'''
    if pd.isna(value):
        return np.nan, False
    s = re.sub(r'[\s\-]', '', str(value)).upper()
    if s == '' or 'X' in s:
        return np.nan, False
    return (s, True) if re.fullmatch(r'\d{12}', s) else (np.nan, False)



**Sanity-check the mapping tables** — every distinct raw value in each messy
categorical column must resolve to something in its map (no silent `NaN`s from
an incomplete lookup table):


In [ ]:

def check_coverage(series, mapping, name):
    vals = series.dropna().apply(lambda x: str(x).strip().upper())
    unmapped = sorted(set(vals) - set(mapping.keys()))
    status = 'OK' if not unmapped else f'MISSING: {unmapped}'
    print(f'{name:22s} -> {status}')

check_coverage(txns_raw['status'], STATUS_MAP, 'txn status')
check_coverage(kyc_raw['kyc_status'], KYC_STATUS_MAP, 'kyc_status')
check_coverage(kyc_raw['risk_segment'], RISK_SEGMENT_MAP, 'risk_segment')
check_coverage(merch_raw['merchant_status'], MERCHANT_STATUS_MAP, 'merchant_status')
check_coverage(merch_raw['business_type'], BUSINESS_TYPE_MAP, 'business_type')
check_coverage(cbk_raw['resolution_status'], RESOLUTION_MAP, 'resolution_status')
check_coverage(cbk_raw['severity'], SEVERITY_MAP, 'severity')
check_coverage(cbk_raw['reason_code'], REASON_CODE_MAP, 'reason_code')
check_coverage(cbk_raw['channel'], CHANNEL_MAP, 'channel')



## 3. Clean each table

### 3.1 De-duplication strategy
Two very different kinds of "duplicate" show up in this dataset, and they need
different treatment:

* **Exact full-row duplicates** (e.g. the same `txn_id` logged twice with identical
  values) — these are safe to drop outright.
* **Same logical entity, conflicting field values** (e.g. the same `user_id` appearing
  twice in KYC with two different `kyc_status`/`signup_timestamp` values, likely from
  re-submitted KYC or overwritten records) — blindly keeping "the first row" or
  "the last row" is arbitrary. Instead we keep, per entity, the **most complete**
  record (most non-null fields) and break ties with the **most recent** date field —
  a defensible, explainable rule rather than silent data loss.


In [ ]:

def dedupe_keep_best(df, id_col, date_col=None):
    '''For entities with multiple conflicting rows after ID normalization: keep the
    most complete row, tie-broken by the most recent value in `date_col`.'''
    d = df.copy()
    d['_completeness'] = d.notna().sum(axis=1)
    d['_date_rank'] = d[date_col].fillna(pd.Timestamp('1900-01-01')) if date_col else 0
    d = d.sort_values(['_completeness', '_date_rank'], ascending=[False, False])
    d = d.drop_duplicates(subset=[id_col], keep='first')
    return d.drop(columns=['_completeness', '_date_rank']).sort_index()


### 3.2 Transactions

In [ ]:

txns = txns_raw.copy()
txns['user_id'] = txns['user_id'].apply(lambda x: normalize_id(x, 'USR', 5))
txns['merchant_id'] = txns['merchant_id'].apply(lambda x: normalize_id(x, 'MCH', 4))
txns['txn_id'] = txns['txn_id'].apply(lambda x: normalize_id(x, 'TXN', 8))
txns['amount_was_negative'] = txns['amount'].apply(was_negative)
txns['amount'] = txns['amount'].apply(clean_amount)
txns['txn_timestamp'] = txns['timestamp'].apply(parse_datetime)
txns['utr_clean'] = txns['utr'].apply(clean_utr)
txns['utr_missing_or_invalid'] = txns['utr_clean'].isna()
txns['mcc_clean'] = txns['mcc'].apply(clean_mcc)
txns['status_clean'] = txns['status'].apply(lambda v: clean_text_upper_map(v, STATUS_MAP))
txns = txns.drop(columns=['timestamp', 'utr', 'mcc', 'status'])

before = len(txns)
txns = txns.drop_duplicates(subset=['txn_id', 'user_id', 'merchant_id', 'amount', 'txn_timestamp', 'utr_clean'])
print(f'Dropped {before - len(txns)} exact duplicate transaction rows -> {len(txns)} remain')

txns = dedupe_keep_best(txns, 'txn_id', 'txn_timestamp').reset_index(drop=True)
print(f'After resolving conflicting duplicate txn_id records: {len(txns)} unique transactions')
print(f'Impossible/unparseable timestamps found: {txns["txn_timestamp"].isna().sum()}')
print(f'Missing/invalid UTRs: {txns["utr_missing_or_invalid"].sum()} ({txns["utr_missing_or_invalid"].mean():.1%})')
print(f'Negative raw amounts corrected to magnitude: {txns["amount_was_negative"].sum()}')
txns.head()


### 3.3 KYC records

In [ ]:

kyc = kyc_raw.copy()
kyc['user_id'] = kyc['user_id'].apply(lambda x: normalize_id(x, 'USR', 5))

pan_parsed = kyc['pan'].apply(clean_pan)
kyc['pan_clean'] = pan_parsed.apply(lambda t: t[0])
kyc['pan_valid'] = pan_parsed.apply(lambda t: t[1])

aad_parsed = kyc['aadhaar'].apply(clean_aadhaar)
kyc['aadhaar_clean'] = aad_parsed.apply(lambda t: t[0])
kyc['aadhaar_valid'] = aad_parsed.apply(lambda t: t[1])

kyc['date_of_birth_clean'] = kyc['date_of_birth'].apply(parse_datetime)
kyc['city_clean'] = kyc['city'].apply(clean_city)
kyc['monthly_income_clean'] = kyc['monthly_income'].apply(clean_amount)
kyc['signup_timestamp_clean'] = kyc['signup_timestamp'].apply(parse_datetime)
kyc['kyc_status_clean'] = kyc['kyc_status'].apply(lambda v: clean_text_upper_map(v, KYC_STATUS_MAP))
kyc['risk_segment_clean'] = kyc['risk_segment'].apply(lambda v: clean_text_upper_map(v, RISK_SEGMENT_MAP))
kyc['full_name'] = kyc['full_name'].str.strip().str.title()
kyc['occupation'] = kyc['occupation'].str.strip().str.title()

kyc = kyc.drop(columns=['pan', 'aadhaar', 'date_of_birth', 'city', 'monthly_income',
                         'signup_timestamp', 'kyc_status', 'risk_segment'])

before = len(kyc)
kyc = dedupe_keep_best(kyc, 'user_id', 'signup_timestamp_clean').reset_index(drop=True)
print(f'KYC: {before} raw rows -> {len(kyc)} unique customers after resolving conflicting duplicates')
print(f'PAN valid rate: {kyc["pan_valid"].mean():.1%}   |   Aadhaar valid rate: {kyc["aadhaar_valid"].mean():.1%}')
kyc.head()


### 3.4 Merchant master

In [ ]:

merch = merch_raw.copy()
merch['merchant_id'] = merch['merchant_id'].apply(lambda x: normalize_id(x, 'MCH', 4))
merch['mcc_clean'] = merch['mcc'].apply(clean_mcc)
merch['merchant_category_clean'] = merch['mcc_clean'].map(MCC_CATEGORY_MAP)
merch['business_type_clean'] = merch['business_type'].apply(
    lambda v: clean_text_upper_map(v, BUSINESS_TYPE_MAP,
                                    default=str(v).strip().title() if pd.notna(v) else np.nan))
merch['city_clean'] = merch['city'].apply(clean_city)
merch['onboarding_date_clean'] = merch['onboarding_date'].apply(parse_datetime)
merch['merchant_status_clean'] = merch['merchant_status'].apply(lambda v: clean_text_upper_map(v, MERCHANT_STATUS_MAP))
merch['avg_ticket_was_negative'] = merch['declared_avg_ticket_size'].apply(was_negative)
merch['declared_avg_ticket_size_clean'] = merch['declared_avg_ticket_size'].apply(clean_amount)
merch['settlement_account_on_file'] = merch['settlement_account'].notna() & (merch['settlement_account'].str.upper() != 'NA')
merch['merchant_name'] = merch['merchant_name'].str.strip().str.replace(r'\s+', ' ', regex=True)

merch = merch.drop(columns=['mcc', 'merchant_category', 'business_type', 'city', 'onboarding_date',
                             'merchant_status', 'declared_avg_ticket_size', 'settlement_account'])

before = len(merch)
merch = dedupe_keep_best(merch, 'merchant_id', 'onboarding_date_clean').reset_index(drop=True)
print(f'Merchants: {before} raw rows -> {len(merch)} unique merchants after resolving conflicting duplicates')
print(f'Merchants missing a settlement account on file: {(~merch["settlement_account_on_file"]).sum()} ({(~merch["settlement_account_on_file"]).mean():.1%})')
merch.head()


### 3.5 Chargebacks / disputes

In [ ]:

cbk = cbk_raw.copy()
cbk['complaint_id'] = cbk['complaint_id'].str.strip().str.upper()
cbk['txn_id'] = cbk['txn_id'].apply(lambda x: normalize_id(x, 'TXN', 8))
cbk['user_id'] = cbk['user_id'].apply(lambda x: normalize_id(x, 'USR', 5))
cbk['merchant_id'] = cbk['merchant_id'].apply(lambda x: normalize_id(x, 'MCH', 4))
cbk['disputed_amount_clean'] = cbk['disputed_amount'].apply(clean_amount)
cbk['transaction_timestamp_clean'] = cbk['transaction_timestamp'].apply(parse_datetime)
cbk['reported_timestamp_clean'] = cbk['reported_timestamp'].apply(parse_datetime)
cbk['bank_response_timestamp_clean'] = cbk['bank_response_timestamp'].apply(parse_datetime)
cbk['resolution_status_clean'] = cbk['resolution_status'].apply(lambda v: clean_text_upper_map(v, RESOLUTION_MAP))
cbk['severity_clean'] = cbk['severity'].apply(lambda v: clean_text_upper_map(v, SEVERITY_MAP))
cbk['reason_code_clean'] = cbk['reason_code'].apply(lambda v: clean_text_upper_map(v, REASON_CODE_MAP))
cbk['channel_clean'] = cbk['channel'].apply(lambda v: clean_text_upper_map(v, CHANNEL_MAP))
cbk['complaint_text'] = cbk['complaint_text'].str.strip()

cbk = cbk.drop(columns=['disputed_amount', 'transaction_timestamp', 'reported_timestamp',
                         'bank_response_timestamp', 'resolution_status', 'severity',
                         'reason_code', 'channel'])

before = len(cbk)
cbk = cbk.drop_duplicates(subset=['complaint_id']).reset_index(drop=True)
print(f'Dropped {before - len(cbk)} exact duplicate complaint rows -> {len(cbk)} remain')

cbk['report_delay_days'] = (cbk['reported_timestamp_clean'] - cbk['transaction_timestamp_clean']).dt.days
impossible_delay = (cbk['report_delay_days'] < 0).sum()
print(f'Disputes reportedly filed *before* the transaction happened (bad data, excluded from delay stats): {impossible_delay}')
cbk.loc[cbk['report_delay_days'] < 0, 'report_delay_days'] = np.nan
cbk.head()



## 4. Foreign-key validation — and a real fraud signal, not just messy data

Joining `transactions.user_id` → `kyc.user_id` and `transactions.merchant_id` →
`merchants.merchant_id` reveals a **large** orphan rate. At first glance this looks
like a data-quality failure — but the numbers don't line up with "typo noise":


In [ ]:

kyc_ids = set(kyc['user_id'].dropna())
merch_ids = set(merch['merchant_id'].dropna())
txn_ids = set(txns['txn_id'].dropna())

txns['has_kyc_match'] = txns['user_id'].isin(kyc_ids)
txns['has_merchant_match'] = txns['merchant_id'].isin(merch_ids)
cbk['txn_fk_valid'] = cbk['txn_id'].isin(txn_ids)
cbk['user_fk_valid'] = cbk['user_id'].isin(kyc_ids)
cbk['merchant_fk_valid'] = cbk['merchant_id'].isin(merch_ids)

n_txn_users = txns['user_id'].nunique()
n_kyc_users = kyc['user_id'].nunique()
overlap_users = len(set(txns['user_id'].dropna()) & kyc_ids)

print(f'Distinct users transacting        : {n_txn_users:,}')
print(f'Distinct users with a KYC record  : {n_kyc_users:,}')
print(f'Users present in BOTH             : {overlap_users:,}')
print(f'-> {1 - overlap_users/n_txn_users:.1%} of transacting users have NO matching KYC record at all\n')

n_txn_merch = txns['merchant_id'].nunique()
n_merch = merch['merchant_id'].nunique()
overlap_merch = len(set(txns['merchant_id'].dropna()) & merch_ids)
print(f'Distinct merchants transacted with : {n_txn_merch:,}')
print(f'Distinct merchants in master file  : {n_merch:,}')
print(f'Merchants present in BOTH          : {overlap_merch:,}')
print(f'-> {1 - overlap_merch/n_txn_merch:.1%} of transacted-with merchants are NOT in the merchant master at all')



Both user IDs and merchant IDs are drawn from the same numeric ID space across
files (e.g. `USR10001`–`USR99999`), and the overlap fraction is almost exactly what
you'd expect from **two independently-sampled subsets of the same ID range**
(≈ KYC's coverage rate of the full range × transaction volume) — not from format
mismatches (those were already fixed by `normalize_id`).

This is a deliberate, and business-relevant, feature of the data, not noise to be
imputed or dropped:

* **Transactions with no matching KYC record** are exactly the profile of
  **synthetic-identity fraud** — money moving through an identity the bank never
  verified.
* **Transactions with no matching merchant-master record** are exactly the profile
  of a **compromised / unregistered merchant account** — a payee that was never
  properly onboarded.

We therefore **keep** these transactions in the fact table (dropping them would
throw away the fraud signal itself) and instead carry `has_kyc_match` /
`has_merchant_match` as flags for downstream risk scoring.


## 5. Analytics-ready data model

In [ ]:

# fact_transactions: transaction grain, enriched with merchant category + customer risk
fact_txn = txns.merge(
    merch[['merchant_id', 'merchant_category_clean', 'merchant_status_clean']],
    on='merchant_id', how='left'
).merge(
    kyc[['user_id', 'kyc_status_clean', 'risk_segment_clean']],
    on='user_id', how='left'
)
# category fallback: if the merchant isn't in the master file, derive category from the txn's own MCC
fact_txn['merchant_category_final'] = fact_txn['merchant_category_clean'].fillna(
    fact_txn['mcc_clean'].map(MCC_CATEGORY_MAP)
)

# fact_chargebacks: dispute grain, enriched via the transaction it disputes
fact_cbk = cbk.merge(
    fact_txn[['txn_id', 'amount', 'merchant_category_final', 'status_clean']].rename(
        columns={'amount': 'txn_amount', 'status_clean': 'txn_status'}
    ),
    on='txn_id', how='left'
)

# --- ADDED: Impute missing dispute amounts with the full transaction amount ---
fact_cbk['disputed_amount_imputed'] = fact_cbk['disputed_amount_clean'].isna()
fact_cbk['disputed_amount_clean'] = fact_cbk['disputed_amount_clean'].fillna(fact_cbk['txn_amount'])

dim_customers = kyc.copy()
dim_merchants = merch.copy()

print('fact_transactions :', fact_txn.shape)
print('fact_chargebacks  :', fact_cbk.shape)
print('dim_customers     :', dim_customers.shape)
print('dim_merchants     :', dim_merchants.shape)
print(f"Dispute amounts imputed from transaction amount: {fact_cbk['disputed_amount_imputed'].sum()}")


## 6. Business metrics

In [ ]:

succ_txn = fact_txn[fact_txn['status_clean'] == 'SUCCESS']

metrics = {
    'Total transaction count': len(fact_txn),
    'Total transaction amount (₹)': round(fact_txn['amount'].sum(), 2),
    'Average transaction value (₹)': round(fact_txn['amount'].mean(), 2),
    'Failed transaction rate': round((fact_txn['status_clean'] == 'FAILED').mean(), 4),
    'Pending transaction rate': round((fact_txn['status_clean'] == 'PENDING').mean(), 4),
    'Chargeback count': len(fact_cbk),
    'Chargeback amount (₹)': round(fact_cbk['disputed_amount_clean'].sum(), 2),
    'Chargeback-to-transaction ratio (count basis, vs successful txns)': round(len(fact_cbk) / len(succ_txn), 4),
    'KYC completion rate (Verified)': round((dim_customers['kyc_status_clean'] == 'VERIFIED').mean(), 4),
    'KYC rejection rate': round((dim_customers['kyc_status_clean'] == 'REJECTED').mean(), 4),
    'Average dispute reporting delay (days)': round(fact_cbk['report_delay_days'].mean(), 2),
    'Transactions with missing/invalid UTR': int(fact_txn['utr_missing_or_invalid'].sum()),
    'Transactions with no KYC match (synthetic-identity risk)': int((~fact_txn['has_kyc_match']).sum()),
    'Transactions with no merchant-master match (unregistered-merchant risk)': int((~fact_txn['has_merchant_match']).sum()),
}

metrics_df = pd.DataFrame(metrics.items(), columns=['Metric', 'Value'])
metrics_df


In [ ]:
os.makedirs('cleaned_data', exist_ok=True)
fact_txn.to_parquet('cleaned_data/fact_transactions.parquet', index=False)
fact_cbk.to_parquet('cleaned_data/fact_chargebacks.parquet', index=False)
dim_customers.to_parquet('cleaned_data/dim_customers.parquet', index=False)
dim_merchants.to_parquet('cleaned_data/dim_merchants.parquet', index=False)
print("Cleaned analytical tables successfully exported!")

In [ ]:
import os

os.makedirs('cleaned_data', exist_ok=True)

tables = {
    'fact_transactions': fact_txn,
    'fact_chargebacks': fact_cbk,
    'dim_customers': dim_customers,
    'dim_merchants': dim_merchants
}

for name, df in tables.items():
    # 1. Save as CSV (easy for evaluators to view on GitHub/Excel)
    df.to_csv(f'cleaned_data/{name}.csv', index=False)
    
    # 2. Save as Parquet (fast & retains perfect data types for Streamlit/AI Agent)
    df.to_parquet(f'cleaned_data/{name}.parquet', index=False)

print("Exported both .csv and .parquet formats successfully!")